# Module 09: Multi-Gaussian Expansion & Linear Light Profiles

## Learning to Autolens

---

**Purpose:** Learn how PyAutoLens uses *linear light profiles* and *Multi-Gaussian
Expansions* (MGEs) to model galaxy light with far fewer non-linear parameters than
traditional Sersic profiles. MGE is now the default approach in PyAutoLens's SLaM
pipeline and is the recommended starting point for all lens modeling.

**Prerequisites:**
- Module 03 (Your First Lens Model) -- fitting workflow, non-linear search
- Module 04 (Search Chaining & SLaM) -- pipeline structure, chaining concept
- Linear algebra basics (matrix inversion, least-squares solutions)

**Key references:**
- Emsellem, Monnet & Bacon (1994), A&A, 285, 723 -- *MGE method for stellar dynamics*
- Cappellari (2002), MNRAS, 333, 400 -- *Efficient MGE fitting* (hereafter **C02**)
- Nightingale, Dye & Massey (2018), arXiv:1708.07377 -- *AutoLens* (hereafter **Nightingale+18**)
- Lawson & Hanson (1974) -- *Solving Least Squares Problems* (NNLS algorithm)

**Companion LaTeX notes:** `../../Notes/09_MGE/09_mge_theory.tex`

---

## Table of Contents

0. [Imports & Data](#0-imports)
1. [Why Linear Light Profiles?](#1-why-linear)
2. [The Inversion: How Intensities Are Solved](#2-inversion)
3. [Basis Functions: Grouping Profiles](#3-basis)
4. [Manual MGE Fit (No Search)](#4-manual-mge)
5. [MGE Modeling with a Non-Linear Search](#5-mge-model)
6. [Running the MGE Fit](#6-running)
7. [The `mge_model_from` Utility](#7-utility)
8. [MGE in the SLaM Pipeline](#8-slam)
9. [Sersic vs MGE: When to Use What](#9-comparison)
10. [Exercises](#10-exercises)

---

## 0. Imports & Data <a id="0-imports"></a>

We load the `simple` dataset, which includes lens light (a Sersic bulge), an
SIE mass model with external shear, and a SersicCore source galaxy. This is the
same dataset used in the SLaM tutorial and provides a realistic test case for MGE.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import autolens as al
import autolens.plot as aplt
import autofit as af

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

print(f"PyAutoLens version: {al.__version__}")

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================
# The 'simple' dataset includes lens light, which is essential
# for demonstrating MGE. The true model is stored in tracer.json.
#
# True lens:  Sersic bulge (n=3, R_e=0.6", I=2.0)
#             Isothermal (theta_E=1.6") + ExternalShear
# True source: SersicCore (n=1, R_e=0.1")
# ============================================================

dataset_path = Path(
    "../../autolens_workspace_latest/dataset/imaging/simple"
)

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,
)
dataset = dataset.apply_mask(mask=mask)

dataset_plotter = aplt.ImagingPlotter(dataset=dataset)
dataset_plotter.subplot_dataset()

---

## 1. Why Linear Light Profiles? <a id="1-why-linear"></a>

### The Problem with `intensity`

In a standard Sersic profile, the `intensity` parameter is highly degenerate with
shape parameters (`effective_radius`, `sersic_index`). The non-linear search wastes
many likelihood evaluations exploring different intensity-shape combinations that
produce nearly identical images.

### The Solution: Solve Intensity Analytically

A *linear light profile* removes `intensity` from the non-linear search entirely.
Instead, given fixed shape parameters, the optimal intensity is computed via linear
algebra in a single step -- no iterative searching needed.

PyAutoLens provides linear versions of all standard light profiles in the
`al.lp_linear` module. Compare:

| Standard (`al.lp`) | Linear (`al.lp_linear`) |
|---------------------|-------------------------|
| `intensity` is a free parameter | `intensity` solved analytically |
| N+1 non-linear params | N non-linear params |
| Degeneracy with shape | No intensity degeneracy |

Let's see this in action.

In [ ]:
# ============================================================
# LINEAR vs STANDARD LIGHT PROFILES
# ============================================================
# A standard Sersic requires intensity as an input parameter.
# A linear Sersic does NOT -- it has a placeholder intensity=1.0
# that gets replaced by the analytically solved value during fitting.
# ============================================================

# Standard Sersic -- must specify intensity
standard_sersic = al.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=(0.05, 0.0),
    intensity=2.0,
    effective_radius=0.6,
    sersic_index=3.0,
)

# Linear Sersic -- no intensity needed
linear_sersic = al.lp_linear.Sersic(
    centre=(0.0, 0.0),
    ell_comps=(0.05, 0.0),
    effective_radius=0.6,
    sersic_index=3.0,
)

print(f"Standard Sersic intensity: {standard_sersic.intensity}")
print(f"Linear Sersic intensity:   {linear_sersic.intensity}  (placeholder -- solved during fit)")
print(f"\nThe linear version has the same shape parameters but intensity")
print(f"is NOT a free parameter -- it's determined by the data.")

In [ ]:
# ============================================================
# QUICK FIT WITH A LINEAR SERSIC
# ============================================================
# Use the true mass model (from tracer.json) and a linear Sersic
# for the lens light. The intensity is solved analytically.
# ============================================================

# True mass model from tracer.json
lens_galaxy = al.Galaxy(
    redshift=0.5,
    bulge=al.lp_linear.Sersic(
        centre=(0.0, 0.0),
        ell_comps=(0.05263, 0.0),
        effective_radius=0.6,
        sersic_index=3.0,
    ),
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        ell_comps=(0.05263, 0.0),
        einstein_radius=1.6,
    ),
    shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05),
)

source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp_linear.SersicCore(
        centre=(0.0, 0.0),
        ell_comps=(0.09622, -0.05556),
        effective_radius=0.1,
        sersic_index=1.0,
        radius_break=0.025,
        alpha=3.0,
        gamma=0.25,
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])
fit = al.FitImaging(dataset=dataset, tracer=tracer)

print(f"chi-squared = {fit.chi_squared:.1f}")
print(f"Reduced chi-squared = {fit.chi_squared / fit.mask.pixels_in_mask:.4f}")
print(f"\nThe linear solver found the optimal intensities automatically!")

fit_plotter = aplt.FitImagingPlotter(fit=fit)
fit_plotter.subplot_fit()

---

## 2. The Inversion: How Intensities Are Solved <a id="2-inversion"></a>

### Theory: The Linear System

Each linear light profile produces an image $\mathbf{b}_j$ (its "blurring matrix
column") when evaluated on the data grid and convolved with the PSF. The total
model image is a weighted sum:

$$
\mathbf{d}_{\text{model}} = \sum_{j=1}^{N} a_j \, \mathbf{b}_j
$$

where $a_j$ are the unknown intensities. In matrix form:

$$
\mathbf{d} = \mathbf{B} \, \mathbf{a} + \mathbf{n}
$$

The maximum-likelihood solution minimizes $\chi^2 = (\mathbf{d} - \mathbf{B}\mathbf{a})^T \mathbf{C}^{-1} (\mathbf{d} - \mathbf{B}\mathbf{a})$:

$$
\hat{\mathbf{a}} = (\mathbf{B}^T \mathbf{C}^{-1} \mathbf{B})^{-1} \, \mathbf{B}^T \mathbf{C}^{-1} \, \mathbf{d}
$$

This is a single matrix solve -- instant, no iteration needed.

### Positive-Only Solver (NNLS)

The standard least-squares solution can produce *negative* intensities, which are
unphysical (galaxies don't emit negative light!). PyAutoLens uses a Non-Negative
Least Squares (NNLS) solver (Lawson & Hanson 1974) to enforce $a_j \geq 0$ for
all components. This prevents the positive-negative "ringing" that plagued earlier
implementations.

### Extracting Solved Intensities

After fitting, the solved intensities are available via
`fit.linear_light_profile_intensity_dict`.

In [ ]:
# ============================================================
# EXTRACTING SOLVED INTENSITIES
# ============================================================
# The fit object stores the analytically solved intensity for
# each linear light profile in a dictionary.
# ============================================================

intensity_dict = fit.linear_light_profile_intensity_dict

# The dictionary keys are the profile objects (with verbose repr strings).
# We iterate over them to print the solved intensities.
for profile, intensity in intensity_dict.items():
    # Extract a short class name for readability
    class_name = type(profile).__name__
    print(f"{class_name}: solved intensity = {intensity:.4f}")

print(f"\nTrue lens intensity:   2.0")
print(f"True source intensity: 4.0")

In [ ]:
# ============================================================
# CONVERTING LINEAR PROFILES TO STANDARD PROFILES
# ============================================================
# For visualization and further analysis, we often need a
# tracer where linear profiles are replaced by standard ones
# with the solved intensity values set.
# ============================================================

tracer_with_intensities = fit.model_obj_linear_light_profiles_to_light_profiles

# Now the lens bulge has a real intensity value
print(f"Lens bulge intensity (solved): {tracer_with_intensities.galaxies[0].bulge.intensity:.4f}")
print(f"Source bulge intensity (solved): {tracer_with_intensities.galaxies[1].bulge.intensity:.4f}")

---

## 3. Basis Functions: Grouping Profiles <a id="3-basis"></a>

### From One Profile to Many

A single Sersic profile captures the global shape of a galaxy, but real galaxies
have structure that deviates from perfect Sersic profiles: isophotal twists,
radially varying ellipticity, boxy/disky isophotes, or distinct bulge+disk components.

The idea behind a *Multi-Gaussian Expansion* (Emsellem+ 1994, Cappellari 2002) is
simple: represent the galaxy's light as a sum of many 2D Gaussians with different
sizes ($\sigma$). Each Gaussian captures emission at a different radial scale.

### Why Gaussians?

Gaussians have three special properties that make them ideal basis functions:

1. **Analytic PSF convolution:** The convolution of two Gaussians is another Gaussian:
   $\sigma_{\text{eff}} = \sqrt{\sigma_g^2 + \sigma_{\text{PSF}}^2}$. This is much
   faster than numerical convolution.

2. **Analytic deprojection:** A 2D Gaussian deprojects to a 3D Gaussian via the Abel
   transform, enabling straightforward dynamical modeling (cf. C02 Sec. 2).

3. **Complete basis:** Any smooth, positive light distribution can be approximated
   arbitrarily well by a sum of Gaussians with appropriate $\sigma$ values.

### The `Basis` Container

PyAutoLens groups multiple linear light profiles into a `Basis` object via
`al.lp_basis.Basis(profile_list=...)`. This treats the entire set as a single
model component.

In [ ]:
# ============================================================
# CREATING A BASIS OF GAUSSIANS
# ============================================================
# We create 30 Gaussians with sigma values log-spaced from
# 0.01" to 3.0" (the mask radius). All share the same centre
# and elliptical components.
# ============================================================

total_gaussians = 30
mask_radius = 3.0

# Log-spaced sigma values from 0.01" to mask_radius
log10_sigma_list = np.linspace(-2, np.log10(mask_radius), total_gaussians)
sigma_list = 10 ** log10_sigma_list

print(f"Number of Gaussians: {total_gaussians}")
print(f"Sigma range: {sigma_list[0]:.4f}\" to {sigma_list[-1]:.4f}\"")
print(f"\nFirst 5 sigma values: {[f'{s:.4f}' for s in sigma_list[:5]]}")
print(f"Last 5 sigma values:  {[f'{s:.4f}' for s in sigma_list[-5:]]}")

In [ ]:
# ============================================================
# VISUALIZE INDIVIDUAL GAUSSIANS
# ============================================================
# Show a selection of Gaussians at different sigma values to
# illustrate how they capture emission at different scales.
# ============================================================

grid_viz = al.Grid2D.uniform(shape_native=(100, 100), pixel_scales=0.05)

# Pick 6 representative Gaussians (indices spread across the range)
indices = [0, 5, 10, 15, 20, 29]

fig, axes = plt.subplots(1, len(indices), figsize=(18, 3))

for ax, idx in zip(axes, indices):
    gaussian = al.lp.Gaussian(
        centre=(0.0, 0.0),
        ell_comps=(0.1, 0.05),
        intensity=1.0,
        sigma=sigma_list[idx],
    )
    image = gaussian.image_2d_from(grid=grid_viz)
    ax.imshow(image.native, origin="lower", cmap="inferno")
    ax.set_title(f"$\\sigma$ = {sigma_list[idx]:.3f}\"", fontsize=10)
    ax.axis("off")

plt.suptitle("Individual Gaussians at Different Scales", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Manual MGE Fit (No Search) <a id="4-manual-mge"></a>

### The Key Insight: Zero Non-Linear Parameters

If we *fix* the centre, elliptical components, and sigma values of all Gaussians,
the only unknowns are the intensities -- which are solved analytically. This means
we can fit an MGE to data with **zero non-linear parameters** and **60 linear
parameters** solved instantly via NNLS.

This is remarkably powerful: we can decompose a galaxy's light into 60 Gaussian
components in a fraction of a second, with no iterative search whatsoever.

We use the true mass model from `tracer.json` (Isothermal + ExternalShear) and
let the MGE handle the lens light.

In [ ]:
# ============================================================
# MANUAL MGE: 60 LINEAR GAUSSIANS, FIXED GEOMETRY
# ============================================================
# All Gaussians share the same centre and elliptical components.
# Sigma values are log-spaced. Intensities are solved by NNLS.
#
# We use ell_comps=(0.1, 0.05) which gives a slightly elliptical
# shape -- close to the true lens light but not identical, to
# show that MGE is flexible enough to still get a good fit.
# ============================================================

total_gaussians = 60
mask_radius = 3.0
log10_sigma_list = np.linspace(-2, np.log10(mask_radius), total_gaussians)

# Build the list of linear Gaussians
bulge_gaussian_list = []
for i in range(total_gaussians):
    gaussian = al.lp_linear.Gaussian(
        centre=(0.0, 0.0),
        ell_comps=(0.05, 0.0),       # Close to true lens ellipticity
        sigma=10 ** log10_sigma_list[i],
    )
    bulge_gaussian_list.append(gaussian)

# Group into a Basis
bulge = al.lp_basis.Basis(profile_list=bulge_gaussian_list)

# Build the tracer with the true mass model
lens = al.Galaxy(
    redshift=0.5,
    bulge=bulge,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        ell_comps=(0.05, 0.0),
        einstein_radius=1.6,
    ),
    shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05),
)

# Also use a linear SersicCore for the source (true model)
source = al.Galaxy(
    redshift=1.0,
    bulge=al.lp_linear.SersicCore(
        centre=(0.0, 0.0),
        ell_comps=(0.096, -0.056),
        effective_radius=0.1,
        sersic_index=1.0,
        radius_break=0.025,
        alpha=3.0,
        gamma=0.25,
    ),
)

tracer = al.Tracer(galaxies=[lens, source])
fit = al.FitImaging(dataset=dataset, tracer=tracer)

print(f"Non-linear parameters: 0")
print(f"Linear parameters:     {total_gaussians + 1}  (60 Gaussians + 1 source)")
print(f"chi-squared:           {fit.chi_squared:.1f}")
print(f"Reduced chi-squared:   {fit.chi_squared / fit.mask.pixels_in_mask:.4f}")

In [ ]:
# ============================================================
# VISUALIZE THE MANUAL MGE FIT
# ============================================================

fit_plotter = aplt.FitImagingPlotter(fit=fit)
fit_plotter.subplot_fit()

In [ ]:
# ============================================================
# INTENSITY-vs-SIGMA DECOMPOSITION
# ============================================================
# Extract the solved intensity of each Gaussian and plot it
# against sigma. This reveals which radial scales contribute
# most to the galaxy's light.
# ============================================================

intensity_dict = fit.linear_light_profile_intensity_dict

# Extract intensities for the lens bulge Gaussians
lens_bulge = fit.tracer.galaxies[0].bulge
intensities = []
sigmas = []
for j, profile in enumerate(lens_bulge.profile_list):
    intensities.append(intensity_dict[profile])
    sigmas.append(10 ** log10_sigma_list[j])

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.bar(range(len(intensities)), intensities, color="steelblue", alpha=0.8)
ax.set_xlabel("Gaussian index (increasing $\\sigma$)", fontsize=12)
ax.set_ylabel("Solved intensity", fontsize=12)
ax.set_title("MGE Decomposition: Intensity per Gaussian", fontsize=13)

# Add a secondary x-axis showing sigma values
ax2 = ax.twiny()
tick_indices = [0, 14, 29, 44, 59]
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(tick_indices)
ax2.set_xticklabels([f"{sigmas[i]:.3f}\"" for i in tick_indices], fontsize=9)
ax2.set_xlabel("$\\sigma$ (arcsec)", fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nNumber of Gaussians with non-zero intensity: {sum(1 for I in intensities if I > 1e-10)}")
print(f"Total intensity: {sum(intensities):.4f}")

In [ ]:
# ============================================================
# CONVERT TO PLOTTABLE TRACER
# ============================================================
# model_obj_linear_light_profiles_to_light_profiles replaces
# all linear profiles with standard profiles that have the
# solved intensity values set.
# ============================================================

tracer_solved = fit.model_obj_linear_light_profiles_to_light_profiles

# We can now plot the lens galaxy's light on its own
tracer_plotter = aplt.TracerPlotter(
    tracer=tracer_solved, grid=dataset.grid
)
tracer_plotter.figures_2d(image=True)

---

## 5. MGE Modeling with a Non-Linear Search <a id="5-mge-model"></a>

### From Fixed to Free Geometry

In Section 4 we fixed all geometric parameters by hand. In practice, we want
the search to find the optimal centre, ellipticity, mass model, and source model.
The MGE's power is that even with free geometry, the search space is much simpler
than a Sersic model because:

1. **Intensities** are still solved analytically (not searched)
2. **Sigma values** are fixed to a log-spaced grid (not searched)
3. Only **centre** and **ell_comps** are free, shared across all Gaussians in a group

### The `gaussian_per_basis` Pattern

To capture a bulge+disk decomposition, we use two groups of Gaussians
(`gaussian_per_basis=2`), each group sharing its own elliptical components.
This gives 2 ell_comps pairs = 4 free parameters, plus 2 centre coordinates
shared across all Gaussians = **6 total non-linear parameters** for the lens light.

Compare this to a Sersic bulge+disk: 2 centres + 2 ell_comps + 2 intensities +
2 effective_radii + 2 sersic_indices = **14 non-linear parameters**.

### Building the Model Manually

Below we build the full MGE model step by step using `af.Model(al.lp_linear.Gaussian)`
with linked priors. This shows exactly how the model is structured before we
introduce the `mge_model_from` shortcut in Section 7.

In [ ]:
# ============================================================
# BUILD MGE MODEL MANUALLY
# ============================================================
# This is the full, explicit pattern. Section 7 shows the
# one-line shortcut, but understanding the manual approach
# is essential for customization.
#
# Structure:
#   Lens:  2 x 30 linear Gaussians (bulge+disk), shared centre
#          Isothermal mass + ExternalShear
#   Source: linear SersicCore
# ============================================================

total_gaussians = 30
gaussian_per_basis = 2
mask_radius = 3.0

# Log-spaced sigma values (fixed, not searched)
log10_sigma_list = np.linspace(-2, np.log10(mask_radius), total_gaussians)

# Shared centre priors (all Gaussians in all groups share these)
centre_0 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
centre_1 = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)

bulge_gaussian_list = []

for j in range(gaussian_per_basis):
    # Create a list of Gaussian models for this group
    gaussian_list = af.Collection(
        af.Model(al.lp_linear.Gaussian) for _ in range(total_gaussians)
    )

    for i, gaussian in enumerate(gaussian_list):
        # Link centres: all Gaussians share the same centre
        gaussian.centre.centre_0 = centre_0
        gaussian.centre.centre_1 = centre_1

        # Link ell_comps: all Gaussians in this group share ell_comps
        # (but different groups have different ell_comps)
        gaussian.ell_comps = gaussian_list[0].ell_comps

        # Fix sigma to the log-spaced grid value
        gaussian.sigma = 10 ** log10_sigma_list[i]

    bulge_gaussian_list += gaussian_list

# Wrap all Gaussians in a Basis model
bulge = af.Model(
    al.lp_basis.Basis,
    profile_list=bulge_gaussian_list,
)

# Mass model
mass = af.Model(al.mp.Isothermal)
shear = af.Model(al.mp.ExternalShear)

# Constrain Einstein radius to a sensible range
mass.einstein_radius = af.UniformPrior(lower_limit=0.5, upper_limit=3.0)

# Lens galaxy
lens = af.Model(
    al.Galaxy, redshift=0.5, bulge=bulge, mass=mass, shear=shear
)

# Source galaxy with linear SersicCore
source = af.Model(
    al.Galaxy, redshift=1.0, bulge=al.lp_linear.SersicCore
)

# Full model
model = af.Collection(
    galaxies=af.Collection(lens=lens, source=source)
)

print(f"Total free (non-linear) parameters: {model.total_free_parameters}")
print(f"  Lens light:  6  (2 centre + 2x2 ell_comps)")
print(f"  Lens mass:   5  (2 centre + 2 ell_comps + 1 einstein_radius)")
print(f"  Shear:       2  (gamma_1, gamma_2)")
print(f"  Source:      {model.total_free_parameters - 13}  (centre, ell_comps, R_e, n, radius_break, alpha, gamma)")
print(f"\nLinear parameters (solved, not searched): {total_gaussians * gaussian_per_basis + 1}")

---

## 6. Running the MGE Fit <a id="6-running"></a>

We now run the Nautilus search with `n_live=75`. Because the MGE removes intensity
and size degeneracies, convergence is typically much faster than a Sersic model
despite having a similar number of free parameters.

**Important:** We set `use_jax=False` on the `AnalysisImaging` object. The linear
algebra inversion is not yet fully compatible with JAX's tracing, so we use NumPy.

In [ ]:
# ============================================================
# RUN THE MGE FIT
# ============================================================
# n_live=75 is sufficient for MGE thanks to the simpler
# parameter space. number_of_cores=1 for notebook stability.
# ============================================================

search = af.Nautilus(
    path_prefix=Path("output/module_09"),
    name="search1_mge_lens_sersic_source",
    n_live=75,
    number_of_cores=1,
)

analysis = al.AnalysisImaging(dataset=dataset, use_jax=False)

result = search.fit(model=model, analysis=analysis)

print(f"\nFit complete!")
print(f"Max log likelihood: {result.log_likelihood:.2f}")
print(f"Best-fit Einstein radius: {result.instance.galaxies.lens.mass.einstein_radius:.3f}\"")

In [ ]:
# ============================================================
# VISUALIZE THE MGE FIT RESULT
# ============================================================

fit_plotter = aplt.FitImagingPlotter(fit=result.max_log_likelihood_fit)
fit_plotter.subplot_fit()

In [ ]:
# ============================================================
# CORNER PLOT
# ============================================================
# The corner plot shows the posterior distributions of the
# non-linear parameters. Note the absence of intensity
# parameters -- they were solved analytically!
# ============================================================

plotter = aplt.NestPlotter(samples=result.samples)
plotter.corner_cornerpy()

---

## 7. The `mge_model_from` Utility <a id="7-utility"></a>

### One-Line Model Creation

The manual model building in Section 5 is explicit but verbose. PyAutoLens provides
a utility function that creates the entire MGE model in one line:

```python
al.model_util.mge_model_from(
    mask_radius=3.0,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
)
```

This returns an `af.Model` of `al.lp_basis.Basis` with:
- `total_gaussians * gaussian_per_basis` linear Gaussians
- Shared centre (uniform or Gaussian prior depending on `centre_prior_is_uniform`)
- Linked ell_comps within each group
- Log-spaced sigma values from 0.01" to `mask_radius`

The result has only **4 free parameters** for `gaussian_per_basis=1` (2 centre +
2 ell_comps) or **6 free parameters** for `gaussian_per_basis=2`.

In [ ]:
# ============================================================
# mge_model_from: THE PRODUCTION SHORTCUT
# ============================================================
# Compare the utility function output with our manual approach.
# ============================================================

# Lens MGE: 2 groups of 20 Gaussians (bulge+disk)
lens_bulge_auto = al.model_util.mge_model_from(
    mask_radius=3.0,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
)

print(f"Lens MGE free parameters: {lens_bulge_auto.total_free_parameters}")
print(f"  (2 centre + 2x2 ell_comps = 6)\n")

# Source MGE: 1 group of 20 Gaussians
source_bulge_auto = al.model_util.mge_model_from(
    mask_radius=3.0,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,  # GaussianPrior for source (less certain)
)

print(f"Source MGE free parameters: {source_bulge_auto.total_free_parameters}")
print(f"  (2 centre + 2 ell_comps = 4)\n")

# Build a complete model using the utility
model_auto = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy,
            redshift=0.5,
            bulge=lens_bulge_auto,
            mass=af.Model(al.mp.Isothermal),
            shear=af.Model(al.mp.ExternalShear),
        ),
        source=af.Model(
            al.Galaxy,
            redshift=1.0,
            bulge=source_bulge_auto,
        ),
    ),
)

print(f"Total model free parameters: {model_auto.total_free_parameters}")
print(f"  Lens light: 6, Mass: 5, Shear: 2, Source: 4 = 17 total")

---

## 8. MGE in the Full SLaM Pipeline <a id="8-slam"></a>

### The Production-Standard Workflow

The PyAutoLens team now recommends **MGE as the default light profile** in the
SLaM pipeline. Here we implement the full 5-stage production pipeline, exactly
as described in `slam_start_here.py` from the latest workspace:

| Stage | Name | What It Does | Free Params |
|-------|------|-------------|-------------|
| 1 | **SOURCE LP** | MGE lens + MGE source + Isothermal mass | ~17 nonlinear |
| 2 | **SOURCE PIX 1** | Replace MGE source with pixelized (adaptive mesh init) | ~10 |
| 3 | **SOURCE PIX 2** | Refine pixelized source with adapt images from Stage 2 | ~3 |
| 4 | **LIGHT LP** | Re-fit MGE lens light with mass+source fixed | ~6 |
| 5 | **MASS TOTAL** | PowerLaw mass with lens light+source fixed | ~10 |

### Key Concepts

- **Adapt images:** After SOURCE LP, we compute a lens-light-subtracted image that
  reveals the lensed source. This "adapt image" tells the pixelized mesh where to
  place more pixels (brighter regions get higher resolution).

- **Positions:** Automatically computed from SOURCE LP results — coordinates where
  multiple images of the source appear. These prevent demagnified solutions in
  the pixelized stages.

- **Chaining:** Each stage passes its results to the next. `result.model` passes
  priors (Gaussian around best-fit); `result.instance` fixes parameters exactly.

### SafeAnalysisImaging

As in Module 05, we wrap `AnalysisImaging` to catch linear algebra exceptions
that can occur for extreme parameter values during the search.


In [ ]:
# ============================================================
# SAFE ANALYSIS WRAPPER
# ============================================================
# Catches linear algebra errors from ill-conditioned inversions
# during the non-linear search. Returns a very low likelihood
# instead of crashing.
# ============================================================

class SafeAnalysisImaging(al.AnalysisImaging):
    """Catches linear algebra errors from ill-conditioned inversions."""
    def log_likelihood_function(self, instance):
        try:
            return super().log_likelihood_function(instance)
        except (np.linalg.LinAlgError, Exception) as e:
            if "singular" in str(e).lower() or "positive definite" in str(e).lower():
                return -1.0e99
            raise

In [ ]:
# ============================================================
# FULL 5-STAGE SLaM PIPELINE WITH MGE
# ============================================================
# This is the production-standard workflow recommended by the
# PyAutoLens team. Each stage builds on the previous results.
#
# Expected total runtime: ~60-120 minutes
# ============================================================

mask_radius = 3.0

# ============================================================
# STAGE 1: SOURCE LP
# ============================================================
# MGE lens light (2 x 20 Gaussians) + MGE source (1 x 20)
# + Isothermal mass + ExternalShear
# This gives us an initial mass model and source reconstruction.
# ============================================================
print("=" * 60)
print("STAGE 1: SOURCE LP — MGE lens + MGE source + Isothermal")
print("=" * 60)

lens_bulge_1 = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
)
source_bulge_1 = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

model_1 = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy, redshift=0.5,
            bulge=lens_bulge_1, disk=None,
            mass=af.Model(al.mp.Isothermal),
            shear=af.Model(al.mp.ExternalShear),
        ),
        source=af.Model(
            al.Galaxy, redshift=1.0, bulge=source_bulge_1,
        ),
    ),
)

search_1 = af.Nautilus(
    path_prefix=Path("output/module_09/slam"),
    name="source_lp[1]",
    n_live=200, number_of_cores=1,
)
analysis_1 = al.AnalysisImaging(dataset=dataset, use_jax=False)
source_lp_result = search_1.fit(model=model_1, analysis=analysis_1)

print(f"\nStage 1 complete!")
print(f"Einstein radius: {source_lp_result.instance.galaxies.lens.mass.einstein_radius:.3f}")

# ============================================================
# STAGE 2: SOURCE PIX 1
# ============================================================
# Replace the MGE source with an adaptive pixelized source.
# Lens light is FIXED from Stage 1. Mass priors from Stage 1.
# Positions are computed automatically from the LP result.
# The adapt image (lens-subtracted) guides the mesh density.
# ============================================================
print("\n" + "=" * 60)
print("STAGE 2: SOURCE PIX 1 — Adaptive pixelized source (init)")
print("=" * 60)

galaxy_image_dict_1 = al.galaxy_name_image_dict_via_result_from(
    result=source_lp_result
)
adapt_images_1 = al.AdaptImages(galaxy_name_image_dict=galaxy_image_dict_1)

mass_2 = al.util.chaining.mass_from(
    mass=source_lp_result.model.galaxies.lens.mass,
    mass_result=source_lp_result.model.galaxies.lens.mass,
    unfix_mass_centre=True,
)

model_2 = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy, redshift=0.5,
            bulge=source_lp_result.instance.galaxies.lens.bulge,
            disk=None,
            mass=mass_2,
            shear=source_lp_result.model.galaxies.lens.shear,
        ),
        source=af.Model(
            al.Galaxy, redshift=1.0,
            pixelization=af.Model(
                al.Pixelization,
                mesh=af.Model(al.mesh.RectangularAdaptDensity, shape=(28, 28)),
                regularization=al.reg.Adapt,
            ),
        ),
    ),
)

search_2 = af.Nautilus(
    path_prefix=Path("output/module_09/slam"),
    name="source_pix[1]",
    n_live=150, number_of_cores=1,
)
analysis_2 = SafeAnalysisImaging(
    dataset=dataset,
    adapt_images=adapt_images_1,
    positions_likelihood_list=[
        source_lp_result.positions_likelihood_from(
            factor=3.0, minimum_threshold=0.2
        )
    ],
    use_jax=False,
)
source_pix_result_1 = search_2.fit(model=model_2, analysis=analysis_2)
print("Stage 2 complete!")

# ============================================================
# STAGE 3: SOURCE PIX 2
# ============================================================
# Refine the pixelized source using adapt images from Stage 2.
# Uses RectangularAdaptImage mesh (adapts to source morphology)
# and Adapt regularization. Mass and lens light are FIXED.
# ============================================================
print("\n" + "=" * 60)
print("STAGE 3: SOURCE PIX 2 — Refined adaptive pixelization")
print("=" * 60)

galaxy_image_dict_2 = al.galaxy_name_image_dict_via_result_from(
    result=source_pix_result_1
)
adapt_images_2 = al.AdaptImages(galaxy_name_image_dict=galaxy_image_dict_2)

model_3 = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy, redshift=0.5,
            bulge=source_lp_result.instance.galaxies.lens.bulge,
            disk=None,
            mass=source_pix_result_1.instance.galaxies.lens.mass,
            shear=source_pix_result_1.instance.galaxies.lens.shear,
        ),
        source=af.Model(
            al.Galaxy, redshift=1.0,
            pixelization=af.Model(
                al.Pixelization,
                mesh=af.Model(al.mesh.RectangularAdaptImage, shape=(28, 28)),
                regularization=al.reg.Adapt,
            ),
        ),
    ),
)

search_3 = af.Nautilus(
    path_prefix=Path("output/module_09/slam"),
    name="source_pix[2]",
    n_live=75, number_of_cores=1,
)
analysis_3 = SafeAnalysisImaging(
    dataset=dataset,
    adapt_images=adapt_images_2,
    use_jax=False,
)
source_pix_result_2 = search_3.fit(model=model_3, analysis=analysis_3)
print("Stage 3 complete!")

# ============================================================
# STAGE 4: LIGHT LP
# ============================================================
# Re-fit the MGE lens light to high accuracy.
# Mass and pixelized source are FIXED from Stages 2-3.
# ============================================================
print("\n" + "=" * 60)
print("STAGE 4: LIGHT LP — Refined MGE lens light")
print("=" * 60)

lens_bulge_4 = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=2,
    centre_prior_is_uniform=True,
)

source_4 = al.util.chaining.source_custom_model_from(
    result=source_pix_result_2, source_is_model=False
)

model_4 = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy, redshift=0.5,
            bulge=lens_bulge_4, disk=None,
            mass=source_pix_result_1.instance.galaxies.lens.mass,
            shear=source_pix_result_1.instance.galaxies.lens.shear,
        ),
        source=source_4,
    ),
)

galaxy_image_dict_4 = al.galaxy_name_image_dict_via_result_from(
    result=source_pix_result_1
)
adapt_images_4 = al.AdaptImages(galaxy_name_image_dict=galaxy_image_dict_4)

search_4 = af.Nautilus(
    path_prefix=Path("output/module_09/slam"),
    name="light[1]",
    n_live=150, number_of_cores=1,
)
analysis_4 = al.AnalysisImaging(
    dataset=dataset, adapt_images=adapt_images_4, use_jax=False,
)
light_result = search_4.fit(model=model_4, analysis=analysis_4)
print("Stage 4 complete!")

# ============================================================
# STAGE 5: MASS TOTAL
# ============================================================
# Fit a PowerLaw mass model (more flexible than Isothermal).
# Lens light FIXED from Stage 4. Source FIXED from Stage 3.
# Positions from Stage 3 prevent demagnified solutions.
# ============================================================
print("\n" + "=" * 60)
print("STAGE 5: MASS TOTAL — PowerLaw mass model")
print("=" * 60)

mass_5 = af.Model(al.mp.PowerLaw)
mass_5 = al.util.chaining.mass_from(
    mass=mass_5,
    mass_result=source_pix_result_1.model.galaxies.lens.mass,
    unfix_mass_centre=True,
)

source_5 = al.util.chaining.source_from(result=source_pix_result_2)

model_5 = af.Collection(
    galaxies=af.Collection(
        lens=af.Model(
            al.Galaxy, redshift=0.5,
            bulge=light_result.instance.galaxies.lens.bulge,
            disk=None,
            mass=mass_5,
            shear=source_pix_result_1.model.galaxies.lens.shear,
        ),
        source=source_5,
    ),
)

galaxy_image_dict_5 = al.galaxy_name_image_dict_via_result_from(
    result=source_pix_result_1
)
adapt_images_5 = al.AdaptImages(galaxy_name_image_dict=galaxy_image_dict_5)

search_5 = af.Nautilus(
    path_prefix=Path("output/module_09/slam"),
    name="mass_total[1]",
    n_live=150, number_of_cores=1,
)
analysis_5 = SafeAnalysisImaging(
    dataset=dataset,
    adapt_images=adapt_images_5,
    positions_likelihood_list=[
        source_pix_result_2.positions_likelihood_from(
            factor=3.0, minimum_threshold=0.2
        )
    ],
    use_jax=False,
)
mass_result = search_5.fit(model=model_5, analysis=analysis_5)

print("\n" + "=" * 60)
print("SLaM PIPELINE COMPLETE!")
print("=" * 60)


In [ ]:
# ============================================================
# VISUALIZE FINAL SLaM RESULT
# ============================================================

fit_plotter = aplt.FitImagingPlotter(fit=mass_result.max_log_likelihood_fit)
fit_plotter.subplot_fit()

print(f"Final model summary:")
print(f"  PowerLaw slope (gamma): {mass_result.instance.galaxies.lens.mass.slope:.3f}")
print(f"  Einstein radius:        {mass_result.instance.galaxies.lens.mass.einstein_radius:.3f}\"")
print(f"  Shear gamma_1:          {mass_result.instance.galaxies.lens.shear.gamma_1:.4f}")
print(f"  Shear gamma_2:          {mass_result.instance.galaxies.lens.shear.gamma_2:.4f}")
print(f"\nThe pipeline ran 5 searches, progressing from a simple")
print(f"Isothermal mass model to a PowerLaw with a free slope.")
print(f"True slope = 2.0 (Isothermal is a special case of PowerLaw).")


---

## Viewing pre-computed results from the Cannon cluster <a id="view-cluster-results"></a>

The full MGE SLaM pipeline (5 stages ending in a `PowerLaw` mass fit) takes ~70 min
on Cannon — impractical to re-run every time. Curated results from the reference
run live under `Modules/09_MGE_Linear_Light_Profiles/results/` and are tracked in git,
one directory per SLaM stage:

| Stage | Meaning |
|-------|---------|
| `source_lp[1]` | Parametric (Sérsic) source with SIE lens mass |
| `source_pix[1]` / `source_pix[2]` | Pixelized source initialize → refine |
| `light[1]` | **MGE** LIGHT LP — fits the lens light using linear Basis Gaussians |
| `mass_total[1]` | Final `PowerLaw` mass model with MGE lens light + pixelized source |

Each stage directory has `fit_subplot.png`, `corner.pdf`, `info.txt`,
`model_results.txt`, `samples.csv`, and `summary.json`. The loader cell below
displays one stage; change the argument to inspect others.


### Reproducing these results on Cannon

The `results/` tree you're reading was produced by **Module 10's `fit_module09.py`**
script running on Harvard FASRC Cannon. To reproduce (or extend) the fit end-to-end:

```bash
# From the repo root (laptop):
bash Modules/10_Cluster_Computing/scripts/push_to_cannon.sh --go       # rsync code + checkpoint
bash Modules/10_Cluster_Computing/scripts/seed_cannon_data.sh --go     # first push only

# On Cannon (login.rc.fas.harvard.edu):
cd /n/holystore01/LABS/hernquist_lab/Lab/$USER/learning_to_autolens
sbatch --export=ALL,MODULE=09 Modules/10_Cluster_Computing/scripts/submit_cannon.slurm

# Back on the laptop, once the job finishes:
bash Modules/10_Cluster_Computing/scripts/pull_from_cannon.sh --go     # pulls into results/
```

See [Module 10](../10_Cluster_Computing/10_cluster_computing.ipynb) Section 7 for
the full runbook (env setup, checkpoint auto-resume, expected runtimes, FASRC
partition / account details).


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, IFrame, display, Markdown

RESULTS_ROOT = Path("results")

def show_result(stage_name):
    stage_dir = RESULTS_ROOT / stage_name
    if not stage_dir.exists():
        print(f"no results directory for stage {stage_name!r}")
        return

    summary = json.loads((stage_dir / "summary.json").read_text())
    display(Markdown(
        f"### `{stage_name}`\n"
        f"- max log-likelihood: **{summary.get('max_log_likelihood'):.2f}**\n"
        f"- log evidence: **{summary.get('log_evidence'):.2f}**\n"
        f"- χ²/N = **{summary.get('chi_squared_per_pixel'):.3f}** "
        f"({summary.get('chi_squared_total'):.0f} / {summary.get('n_unmasked_pixels')} unmasked px)\n"
        f"- max |normalized residual| = **{summary.get('max_abs_normalized_residual'):.2f} σ**"
    ))

    fit_png = stage_dir / "fit_subplot.png"
    if fit_png.exists():
        display(Markdown("**Fit subplot** (data, model, residuals, normalized residuals):"))
        display(Image(filename=str(fit_png)))

    corner_pdf = stage_dir / "corner.pdf"
    if corner_pdf.exists():
        display(Markdown("**Posterior corner plot:**"))
        display(IFrame(src=str(corner_pdf), width=720, height=720))

    mr = stage_dir / "model_results.txt"
    if mr.exists():
        preview = "\n".join(mr.read_text().splitlines()[:40])
        display(Markdown(f"**`model_results.txt`** (first 40 lines):\n\n```\n{preview}\n```"))

# Default: the final PowerLaw mass model — the full pipeline's publishable result.
show_result("mass_total[1]")


---

## 9. Sersic vs MGE: When to Use What <a id="9-comparison"></a>

### Decision Guide

| Criterion | Sersic | MGE |
|-----------|--------|-----|
| **Flexibility** | Limited to smooth, symmetric profiles | Captures isophotal twists, radial ellipticity variation |
| **Non-linear parameters** | ~7 per component (intensity, R_e, n, centre, ell_comps) | ~3 per group (centre, ell_comps); intensity + sigma fixed |
| **Parameter degeneracies** | Strong (intensity vs R_e vs n) | Minimal (no intensity/size degeneracy) |
| **Physical interpretability** | Direct (n, R_e are physical) | Indirect (must integrate Gaussians for R_e, total flux) |
| **Speed per likelihood** | Fast (~0.01s for 2 components) | Slower (~0.5s for 60 Gaussians) |
| **Convergence speed** | Slower (complex parameter space) | Faster (simpler parameter space) |
| **Data quality needed** | Works at any resolution | Best for high-resolution (HST, JWST, AO) |
| **Best for** | Quick fits, low-res data, when physical params needed | Production pipelines, complex galaxies, SLaM |

### When MGE is NOT Ideal

- **Low-resolution data** (ground-based seeing-limited): Few pixels on the galaxy
  means a simple Sersic is sufficient and more interpretable.
- **When you need physical Sersic parameters**: If your science requires $n_{\text{Sersic}}$
  or $R_e$ directly (e.g., for scaling relations), a Sersic model is more natural.
  You can still extract an effective $R_e$ from an MGE by integrating the Gaussian
  decomposition, but it's an extra step.
- **Very simple galaxy morphology**: If the galaxy is well-described by a single
  Sersic, the MGE's extra flexibility is unnecessary overhead.

### Recommendation

For most lens modeling applications with space-based data, **start with MGE**. It is
the default in the SLaM pipeline for good reason: it reduces the non-linear parameter
space, eliminates the worst degeneracies, and captures complex morphology that Sersic
profiles miss.

---

## 10. Exercises <a id="10-exercises"></a>

### Exercise 1: Gaussian Count Experiment

Repeat the manual MGE fit (Section 4) with 10, 30, and 60 Gaussians. For each:
- Record the $\chi^2$ and the number of Gaussians with non-zero intensity.
- Time the fit using `%%timeit` or `time.time()`.
- Plot the residuals side-by-side.

At what Gaussian count does the fit quality plateau? Is there a point of diminishing
returns? How does the fit time scale with Gaussian count?

### Exercise 2: Manual vs `mge_model_from`

Build the same MGE model (20 Gaussians, `gaussian_per_basis=1`) both manually
(as in Section 5) and using `al.model_util.mge_model_from()` (Section 7). Verify:
- Both have the same number of free parameters.
- Both produce `model.info` strings with the same structure.
- Run a short Nautilus search (n_live=50, a few iterations) with each and confirm
  the log-likelihoods are comparable.

### Exercise 3: MGE Lens + Pixelized Source

Combine MGE lens light (from this module) with a pixelized source reconstruction
(from Module 05). Set up a two-search chain:
1. Search 1: MGE lens + MGE source + Isothermal (as in Section 6)
2. Search 2: Fix MGE lens light and mass from Search 1, switch source to
   `al.Pixelization` with `Delaunay` mesh and `Constant` regularization.

Compare the source reconstruction from the MGE source vs the pixelized source.
When would you prefer each approach?

### Exercise 4: MGE Decomposition Analysis

Using the manual MGE fit from Section 4:
1. Extract the solved intensity and sigma for each Gaussian.
2. Compute the total flux of each Gaussian: $F_j = 2\pi \, a_j \, \sigma_j^2 \, q_j$,
   where $q_j$ is the axis ratio.
3. Compute the cumulative flux as a function of radius.
4. Find the half-light radius $R_{1/2}$ (the radius enclosing 50% of the total flux).
5. Compare with the true Sersic $R_e = 0.6"$. How close is the MGE-derived value?

*Hint: For a 2D Gaussian with semi-major axis $\sigma$ and axis ratio $q$, the flux*
*enclosed within elliptical radius $r$ is $F(r) = F_{\text{tot}} \, [1 - \exp(-r^2 / 2\sigma^2)]$.*

---

## Summary

| Concept | Implementation | Key Advantage |
|---------|---------------|---------------|
| Linear light profiles | `al.lp_linear.Sersic`, `al.lp_linear.Gaussian` | Intensity solved analytically, not searched |
| Basis functions | `al.lp_basis.Basis(profile_list=...)` | Groups multiple profiles into one component |
| Manual MGE | List of `al.lp_linear.Gaussian` with shared geometry | 0 non-linear params, instant fit |
| MGE modeling | `af.Model(al.lp_linear.Gaussian)` with linked priors | 6 non-linear params for bulge+disk |
| `mge_model_from` | `al.model_util.mge_model_from(...)` | One-line model creation |
| MGE SLaM | 3-stage pipeline: SOURCE LP, LIGHT LP, MASS TOTAL | Production-ready automated fitting |

**Key takeaway:** MGE is the recommended default for lens light modeling in PyAutoLens.
It captures complex galaxy morphology with far fewer non-linear parameters than Sersic
profiles, leading to faster convergence and more reliable fits.

**References:**
- Emsellem, Monnet & Bacon (1994), A&A, 285, 723
- Cappellari (2002), MNRAS, 333, 400
- Nightingale, Dye & Massey (2018), arXiv:1708.07377

**Next module:** We will explore how to prepare and model real observational data --
from raw FITS files to a complete lens model.

---

*Learning to Autolens -- Module 09*
*Rodrigo Cordova Rosado, Harvard CfA*
*Built with Claude Code*